# 摘要评估器

### 设置

In [ ]:
# 您可以在代码中直接设置
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["LANGSMITH_API_KEY"] = ""
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langsmith-academy"

In [ ]:
# 或者您可以使用 .env 文件
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

### 任务

我们这里的任务是分析随机语句的毒性，将它们分类为 `有毒` 或 `无毒`。

查看我们的数据集！

In [ ]:
from langsmith import Client

client = Client()
dataset = client.clone_public_dataset(
    "https://smith.langchain.com/public/89ef0d44-a252-4011-8bb8-6a114afc1522/d"
)

这是一个简单的毒性分类器！

In [ ]:
from openai import OpenAI
openai_client = OpenAI()
from pydantic import BaseModel, Field

class Toxicity(BaseModel):
    toxicity: str = Field(description="""如果该语句有毒则为'有毒'，如果该语句无毒则为'无毒'。""")

def good_classifier(inputs: dict) -> dict:
    completion = openai_client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": f"这是语句: {inputs['statement']}"
            }
        ],
        response_format=Toxicity,
    )

    toxicity_score = completion.choices[0].message.parsed.toxicity
    return {"class": toxicity_score}

### 摘要评估器

这些是摘要评估器函数可以访问的字段：
- `inputs: list[dict]`：来自我们数据集中示例的输入列表
- `outputs: list[dict]`：从在每个输入上运行我们的目标产生的字典输出列表
- `reference_outputs: list[dict]`：来自我们数据集中示例的参考输出列表
- `runs: list[Run]`：从在数据集上运行我们的目标获得的运行对象列表
- `examples: list[Example]`：完整数据集示例的列表，包括示例输入、输出（如果可用）和元数据（如果可用）

现在我们将定义我们的摘要评估器！在这里，我们将计算 f1 分数，这是精确度和召回率的结合。

这种指标只能在我们实验中的所有示例上计算，所以我们的评估器接收输出列表和参考输出列表。

In [ ]:
def f1_score_summary_evaluator(outputs: list[dict], reference_outputs: list[dict]) -> dict:
    true_positives = 0
    false_positives = 0
    false_negatives = 0
    for output_dict, reference_output_dict in zip(outputs, reference_outputs):
        output = output_dict["class"]
        reference_output = reference_output_dict["class"]
        if output == "有毒" and reference_output == "有毒":
            true_positives += 1
        elif output == "有毒" and reference_output == "无毒":
            false_positives += 1
        elif output == "无毒" and reference_output == "有毒":
            false_negatives += 1

    if true_positives == 0:
        return {"key": "f1_score", "score": 0.0}

    precision = true_positives / (true_positives + false_positives)
    recall = true_positives / (true_positives + false_negatives)
    f1_score = 2 * (precision * recall) / (precision + recall)
    return {"key": "f1_score", "score": f1_score}

注意我们将 `f1_score_summary_evaluator` 作为摘要评估器传入！

In [ ]:
results = client.evaluate(
    good_classifier,
    data=dataset,
    summary_evaluators=[f1_score_summary_evaluator],
    experiment_prefix="良好分类器"
)